# 03. Stacje IMGW

Jedna stacja (Warszawa), jeden rok (2012). Bez konta.

In [ ]:
import io
import zipfile

import requests
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://danepubliczne.imgw.pl/data/dane_pomiarowo_obserwacyjne/dane_meteorologiczne/terminowe/synop"

In [ ]:
# naglowek jest w osobnym pliku, dane bez naglowka, kodowanie cp1250
columns = requests.get(f"{BASE}/s_t_nag%C5%82%C3%B3wek.csv").content.decode("cp1250").strip().split(",")

# pomiary godzinowe: Warszawa 2012
content = requests.get(f"{BASE}/2012/2012_375_s.zip").content
with zipfile.ZipFile(io.BytesIO(content)) as archive:
    df = pd.read_csv(archive.open(archive.namelist()[0]), header=None, names=columns, encoding="cp1250", low_memory=False)

df["ts_utc"] = pd.to_datetime(dict(year=df["ROK"], month=df["MC"], day=df["DZ"], hour=df["GG"]))
print(len(df), "wierszy,", len(columns), "kolumn")
df[["POST", "ts_utc", "TEMP", "TGMI", "TMIN", "PKSN", "WO6G"]].head()

Każda wartość ma kolumnę statusu `W...`: `8` = brak pomiaru. Część pomiarów jest tylko o wybranych godzinach:

In [ ]:
measured = pd.DataFrame({col: pd.to_numeric(df[f"W{col}"], errors="coerce") != 8 for col in ["TEMP", "TGMI", "TMIN", "PKSN", "WO6G"]})
measured.groupby(df["ts_utc"].dt.hour).mean().round(2).T

In [ ]:
winter = df[(df["ts_utc"] >= "2012-01-25") & (df["ts_utc"] < "2012-02-20")].set_index("ts_utc")
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
winter["TEMP"].plot(ax=ax[0], label="powietrze")
winter["TGMI"][winter["GG"] == 6].plot(ax=ax[0], style="o", label="min przy gruncie (noc)")
ax[0].axhline(-18, color="red", ls="--")
ax[0].legend()
winter["PKSN"][winter["GG"] == 6].plot(ax=ax[1], style="o-", title="śnieg [cm]")
plt.show()

In [ ]:
# statusy i kody sniegu
for col in ["TEMP", "TGMI", "PKSN"]:
    print(col, pd.to_numeric(df[f"W{col}"], errors="coerce").value_counts(dropna=False).to_dict())
pksn = pd.to_numeric(df["PKSN"], errors="coerce")
print("kody śniegu:", pksn[pksn >= 997].value_counts().to_dict())


In [ ]:
# utc?
summer = df[df["MC"].isin([6, 7, 8])]
print(summer.groupby("GG")["TEMP"].mean().round(1).to_dict())
print("najcieplejsza godzina:", summer.groupby("GG")["TEMP"].mean().idxmax())


In [ ]:
# 2010 i 2011
def load(year):
    content = requests.get(f"{BASE}/{year}/{year}_375_s.zip").content
    with zipfile.ZipFile(io.BytesIO(content)) as archive:
        return pd.read_csv(archive.open(archive.namelist()[0]), header=None, names=columns, encoding="cp1250", low_memory=False)

for year, d in [(2010, load(2010)), (2011, load(2011)), (2012, df)]:
    share = {c: round((pd.to_numeric(d[f"W{c}"], errors="coerce") != 8).mean(), 2) for c in ["TEMP", "TGMI", "PKSN"]}
    print(year, len(d), "wierszy", share)
